# Moukthika — Model Training & Optimization

Finalizable training for Safe-Guard prompt-injection detection.

**Optimization techniques (this notebook / `src/training`):**
- Balanced **class weights** for the injection class
- Validation **threshold tuning** that prioritizes recall (missed attacks are costly)
- **Early stopping** on the MLP path

**Out of scope here:** broad model-selection bakeoff and hyperparameter search (sibling workstreams).

**Dataset:** [`xTRam1/safe-guard-prompt-injection`](https://huggingface.co/datasets/xTRam1/safe-guard-prompt-injection)  
Official `test` split is used **only** for final evaluation; validation is carved from `train`.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.training.train import run_train_eval
from src.training.metrics import format_metrics_report

## Train logistic regression (class weights + threshold tuning)

CLI equivalent:

```bash
PYTHONPATH=. python -m src.training --model logreg --output-dir artifacts/training
```

In [ ]:
out = ROOT / "artifacts" / "training"
report_lr = run_train_eval(
    model_name="logreg",
    min_precision=0.80,
    output_dir=out,
)
test_tuned = report_lr["metrics"]["test_tuned"]
print()
print(format_metrics_report("OFFICIAL TEST (logreg, tuned threshold)", test_tuned))

## Train MLP (early stopping + sample weights + threshold tuning)

```bash
PYTHONPATH=. python -m src.training --model mlp --output-dir artifacts/training
```

In [ ]:
report_mlp = run_train_eval(
    model_name="mlp",
    min_precision=0.80,
    output_dir=out,
)
test_tuned_mlp = report_mlp["metrics"]["test_tuned"]
print()
print(format_metrics_report("OFFICIAL TEST (mlp, tuned threshold)", test_tuned_mlp))
print("MLP early-stopping iters:", report_mlp["model"]["extra"].get("n_iter_"))

## Compare reported test metrics

False-negative rate = FN / (FN + TP) on the injection class. Lower is better for security.

In [ ]:
rows = []
for name, report in (("logreg", report_lr), ("mlp", report_mlp)):
    m = report["metrics"]["test_tuned"]
    thr = report["threshold_tuning"]["selected_threshold"]
    rows.append(
        {
            "model": name,
            "threshold": thr,
            "precision": round(m["precision"], 4),
            "recall": round(m["recall"], 4),
            "f1": round(m["f1"], 4),
            "false_negative_rate": round(m["false_negative_rate"], 4),
            "fn": m["confusion_matrix"]["fn"],
            "fp": m["confusion_matrix"]["fp"],
        }
    )

import pandas as pd
display(pd.DataFrame(rows))

# Persist a compact summary next to JSON metrics
summary_path = out / "test_metrics_summary.json"
summary_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
print("Wrote", summary_path)